# Loading Dependencies

In [ ]:
from base64 import b64encode
import numpy, torch, os
from IPython.display import HTML
import matplotlib.pyplot as plt
from torch import autocast
from torchvision import transforms as tfms
from tqdm.auto import tqdm
from PIL import Image
from kaggle_secrets import UserSecretsClient
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    LMSDiscreteScheduler
)

# Set device

In [ ]:
if torch.cuda.is_available():
  torch_device = "cuda"
elif torch.backends.mps.is_available():
  torch_device = "mps"
  os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = "1"
else: torch_device = "cpu"

# Looking inside the standard pipeline

- This code comes from the [Huggingface example notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb).
- The inference pipeline is just a small piece of code that plugs the components together and performs the inference loop. [This is all there is to it](https://github.com/huggingface/diffusers/blob/main/src/diffusers/pipelines/stable_diffusion/pipeline_stable_diffusion.py#L204).
- We'll go through the process of loading and plugging the pieces to see how we could have written it ourselves.
- We'll start by loading all the modules that we need from their pretrained weights.

## Text encoder and the tokenizer.
- These come from the text portion of a standard CLIP model, so we'll use the weights released by Open AI.

In [ ]:
clip_checkpoint = "openai/clip-vit-large-patch14"
tokenizer = CLIPTokenizer.from_pretrained(
    clip_checkpoint,
    dtype=torch.bfloat16
)
text_encoder = CLIPTextModel.from_pretrained(
    clip_checkpoint,
    dtype=torch.bfloat16,
    device_map = torch_device
)

## The `vae` and the `unet`.
- These are distinct models whose weights are stored inside folders of the Stable Diffusion repository.
- We can use the `subfolder` argument to refer to [these locations](https://huggingface.co/CompVis/stable-diffusion-v1-4/tree/main).
- Here we use a different VAE from the original release, which has been fine-tuned for more steps

In [ ]:
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-ema",
    torch_dtype=torch.bfloat16,
    device_map = torch_device
)
unet = UNet2DConditionModel.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    subfolder="unet",
    torch_dtype=torch.bfloat16,
    device_map = torch_device
)

## The scheduler

During training
- We add some noise to an image and then have the model try to predict the noise.
- If we always added a ton of noise, the model might not have much to work with.
- If we only add a tiny amount, the model won't be able to do much with the random starting points we use for sampling.
- So during training, the amount is varied, according to some distribution.

During sampling
- We want to 'denoise' over a number of steps.
- How many steps and how much noise we should aim for at each step are going to affect the final result.
- The scheduler is in charge of handling all of these details.
- When we want to sample over a smaller number of steps, we set this up with `scheduler.set_timesteps`
- We'll start at a high noise level (in fact, our input will be pure noise) and gradually 'denoise' down to an image, according to this schedule.

Example
- The standard pipeline uses the [PNDM Scheduler](https://arxiv.org/abs/2202.09778)
- To make things a bit different, we'll use another scheduler.
- We'll use [Katherine Crowson's](https://github.com/crowsonkb) excellent K-LMS scheduler.
- We need to be careful to use the same noising schedule that was used during training.
- The schedule is defined by the **number of noising steps** and the **amount of noise added at each step**, which is derived from the `beta` parameters.
- In the case of the k-LMS scheduler, this is how the betas evolve during the 1000 steps of the noising process used during training:

In [ ]:
beta_start,beta_end = 0.00085,0.012

In [ ]:
plt.plot(torch.linspace(beta_start**0.5, beta_end**0.5, 1000) ** 2)
plt.xlabel('Timestep')
plt.ylabel('β');

In [ ]:
scheduler = LMSDiscreteScheduler(
    beta_start=beta_start,
    beta_end=beta_end,
    beta_schedule="scaled_linear",
    num_train_timesteps=1000
)

## Reconstruct the pipeline
- We now define the parameters we'll use for generation.
- In contrast with the previous examples, we set `num_inference_steps` to 70 to get an even more defined image.
- Again, this is adapted from the [HF notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb) and looks very similar to what you'll find if you inspect [the `__call__()` method of the stable diffusion pipeline](https://github.com/huggingface/diffusers/blob/main/src/diffusers/pipelines/stable_diffusion/pipeline_stable_diffusion.py#L200).  

In [ ]:
prompt = ["a photograph of an astronaut riding a horse"]
# prompt = ["A watercolor painting of an otter"]
height = 512
width = 512
num_inference_steps = 70
guidance_scale = 7.5
batch_size = 1

### Create text embeddings

- We tokenize the prompt.
- The model requires the same number of tokens for every prompt,
    - Padding is used to ensure we meet the required length.
- The text encoder gives us the embeddings for the text prompt we used.
- We also get the embeddings required to perform **unconditional generation**, which is achieved with an empty string:
    - The model is free to go in whichever direction it wants as long as it results in a reasonably-looking image.
    - These embeddings will be applied to apply **classifier-free guidance**.
- For **classifier-free guidance**, we need to do two forward passes.
    - One with the conditioned input (`text_embeddings`),
    - and another with the unconditional embeddings (`uncond_embeddings`).
- In practice, we can concatenate both into a single batch to avoid doing two forward passes.

In [ ]:
def create_text_emb(prompt):
    text_input = tokenizer(
      prompt,
      padding="max_length",
      max_length=tokenizer.model_max_length,
      truncation=True,
      return_tensors="pt"
    )
    
    with torch.no_grad():
      text_embeddings = text_encoder(
          text_input.input_ids.to(torch_device)
      ).get('last_hidden_state')
    
    uncond_input = tokenizer(
      [""] * batch_size,
      padding="max_length",
      max_length=text_input.input_ids.shape[-1],
      return_tensors="pt"
    )
    with torch.no_grad():
      uncond_embeddings = text_encoder(
          uncond_input.input_ids.to(torch_device)
      ).get('last_hidden_state')
    
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
    
    return text_input, text_embeddings

In [ ]:
text_input, text_embeddings = create_text_emb(prompt)

print(f"tokenizer output: {text_input}")
print(f"input_ids shape: {text_input['input_ids'].shape}")
print(f"attn mask shape: {text_input['attention_mask'].shape}")
print(f"text_embeddings.shape: {text_embeddings.shape}")
print(f"text_embeddings.dtype: {text_embeddings.dtype}")

### Initialize latents

- To start the denoising process, we start from pure Gaussian (normal) noise. These are our initial latents.
- `4×64×64` is the input shape.
- The decoder will later transform this latent representation into a `3×512×512` image after the denoising process is complete.
- Next, we initialize the scheduler with our chosen `num_inference_steps`.
- This will prepare the internal state to be used during denoising.
- We scale the initial noise by the standard deviation required by the scheduler.
    - This value will depend on the particular scheduler we use.

In [ ]:
latents = torch.randn(
    (batch_size,unet.config.in_channels,height // 8,width // 8),
    generator=torch.manual_seed(100)
)
latents = latents.to(torch_device).to(torch.bfloat16)
print(f"latents.shape: {latents.shape}")
print(f"latents.dtype: {latents.dtype}")

scheduler.set_timesteps(num_inference_steps)
print(f"scheduler.init_noise_sigma: {scheduler.init_noise_sigma}")
latents = latents * scheduler.init_noise_sigma

### Denoising loop
- The timesteps go from `999` to `0` (1000 steps that were used during training) following a particular schedule.
- We expand the latents to apply classifier-free guidanceto without doing two forward passes. `torch.cat([latents] * 2)`

In [ ]:
plt.plot(scheduler.timesteps, scheduler.sigmas[:-1]);
plt.xlabel("timestamps");
plt.ylabel("sigmas");

In [ ]:
def denoise_loop(scheduler, latents, text_embeddings, start_step= 0):
    with autocast(torch_device, dtype=torch.bfloat16):
        for i, t in enumerate(tqdm(scheduler.timesteps)):
            if i >= start_step:
                # initiate and scale latents =====
                latent_model_input = torch.cat([latents] * 2)
                latent_model_input = scheduler.scale_model_input(latent_model_input, t)
                # # for Diffusers 0.3 and below
                # sigma = scheduler.sigmas[i]
                # latent_model_input = latent_model_input/((sigma**2 + 1) ** 0.5)
        
                # predict the noise residual =====
                with torch.no_grad():
                    noise_pred = unet(
                        latent_model_input,
                        timestep = t,
                        encoder_hidden_states=text_embeddings
                    ).sample
        
                # perform guidance =====
                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                noise_pred = (
                    noise_pred_uncond
                    + guidance_scale * (noise_pred_text - noise_pred_uncond)
                )
        
                # compute the previous noisy sample x_t -> x_t-1
                latents = scheduler.step(noise_pred, t, latents).prev_sample
                # # for Diffusers 0.3 and below -> ...["prev_sample"]
    return latents

In [ ]:
latents = denoise_loop(scheduler, latents, text_embeddings)

### Decode denoised latents and produce the final image
- After this process complets our `latents` contain the denoised representation of the image. We use the `vae` decoder to convert it back to pixel space.
- And finally, we convert the image to PIL so we can display it.

In [ ]:
with torch.no_grad(): image = vae.decode(1 / 0.18215 * latents).sample
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).to(torch.float32).numpy()
image = (image * 255).round().astype("uint8")
Image.fromarray(image[0])

- Other diffusion models may be trained with different noising and scheduling approaches, some of which keep the variance fairly constant across noise levels ('variance preserving') with different scaling and mixing tricks instead of having noisy latents with higher and higher variance as more noise is added ('variance exploding').

- The scaling(`latents = latents * scheduler.init_noise_sigma`) and pre-conditioning(`scheduler.scale_model_input` inside the loop) differ between papers and implementations, so keep an eye out for this if you work with a different type of diffusion model.

# Looking inside an img2img pipeline

## Download a demo image

In [ ]:
sample_image_path = 'https://lafeber.com/pet-birds/wp-content/uploads/2018/06/Scarlet-Macaw-2.jpg'
!curl --output macaw.jpg {sample_image_path}

## Experiment with the `vae`

In [ ]:
def pil_to_latent(input_im):
    # Single image -> single latent in a batch with size (1, 4, H//8, W//8)
    with torch.no_grad():
        tensor_input = tfms.ToTensor()(input_im).unsqueeze(0).to(torch_device).to(torch.bfloat16)
        print(f"tensor_input.dtype: {tensor_input.dtype}")
        latent = vae.encode(tensor_input*2-1) # scaling from [0,1] --> [-1,1]
    return 0.18215 * latent.latent_dist.sample()

def latents_to_pil(latents):
    # bath of latents -> list of images
    latents = (1 / 0.18215) * latents
    with torch.no_grad():
        image = vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).to(torch.float32).numpy()
    images = (image * 255).round().astype("uint8")
    pil_images = [Image.fromarray(image) for image in images]
    return pil_images

In [ ]:
# Load the image with PIL
input_image = Image.open('macaw.jpg').resize((512, 512))
print("original image")
display(input_image)

# Encode to the latent space
encoded = pil_to_latent(input_image)
print(f"encoded latent shape: {encoded.shape}")

# Let's visualize the four channels of this latent representation:
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for c in range(4):
    axs[c].imshow(encoded[0][c].to(torch.float32).cpu(), cmap='Greys')

# Decode this latent representation back into an image
decoded = latents_to_pil(encoded)[0]
display(decoded)
print("decoded image")

## Create the image to start denoising from

- Instead of pure noise (`torch.randn` --> multiplied by `scheduler.init_noise_sigma`), We're going to add noise to our encoded image and use it as the starting image.
- you can run `??scheduler.add_noise` to see that `scheduler.add_noise` function literally adds noise scaled by sigma: `noisy_samples = original_samples + noise * sigmas`
- We'll use a similar loop to the first demo, but we'll skip the first `start_step` steps.
- To noise our image, we'll use the scheduler to noise it to a level equivalent to `start_step`.

to get further away from the input image.
- add more noise
- do more denoising steps

In [ ]:
num_inference_steps = 40
start_step = 5
noise = torch.randn_like(encoded) # Random noise

scheduler.set_timesteps(num_inference_steps)
noise = torch.randn_like(encoded)
latents = scheduler.add_noise(
    encoded,
    noise,
    timesteps=torch.tensor([scheduler.timesteps[start_step]])
).to(torch_device)

- Let's visualize what this looks like if we decode this noisy version using the `vae`

In [ ]:
latents_to_pil(latents)[0]

## Denoising loop

In [ ]:
prompt = ["A colorful dancer, nat geo photo"]
guidance_scale = 8
_, text_embeddings = create_text_emb(prompt)

In [ ]:
latents = denoise_loop(scheduler, latents, text_embeddings, start_step=start_step)
latents_to_pil(latents)[0]

# Looking inside the text -> embedding pipeline

In [ ]:
prompt = 'A picture of a puppy'
# Turn the text into a sequence of tokens:
text_input = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)

# print(text_input['input_ids'][0])
# # See the individual tokens
# for t in text_input['input_ids'][0][:7]:
#     print(t, tokenizer.decoder.get(int(t)))

input_ids = text_input.input_ids.to(torch_device)
encoder_out = text_encoder(input_ids)
print(encoder_out.keys())
output_embeddings = encoder_out['last_hidden_state']
print('output_embeddings.shape:', output_embeddings.shape)
output_embeddings

To replicate these embeddings, there are actually two steps - as revealed by inspecting `text_encoder.text_model.embeddings`:

In [ ]:
text_encoder.text_model.embeddings

## Token embeddings

In [ ]:
token_emb_layer = text_encoder.text_model.embeddings.token_embedding
print("the token emb layer:")
display(token_emb_layer) # (Vocab size , emb_dim)

print("\n\ntoken embedding for the whole prompt")
token_embeddings = token_emb_layer(text_input.input_ids.to(torch_device))
print(token_embeddings.shape) # (bsz, seq_len, emb_dim)
token_embeddings

## Position embeddings

Positional embeddings tell the model where in a sequence a token is. Much like the token embedding, this is a set of (optionally learnable) parameters. But now instead of dealing with ~50k tokens we just need one for each position (77 total):

In [ ]:
pos_emb_layer = text_encoder.text_model.embeddings.position_embedding
print("the positional embedding layer")
display(pos_emb_layer)

position_ids = text_encoder.text_model.embeddings.position_ids[:, :77]
position_embeddings = pos_emb_layer(position_ids)
print("\npositional embedding for each position")
print(position_embeddings.shape)
display(position_embeddings)

In [ ]:
input_embeddings = token_embeddings + position_embeddings
input_embeddings2 = text_encoder.text_model.embeddings(
    text_input.input_ids.to(torch_device)
)

print((input_embeddings == input_embeddings2).all())

## Attnetion + encoder + layerNorm

In [ ]:
from transformers.modeling_attn_mask_utils import \
_create_4d_causal_attention_mask

# start with these two to understand the logic of this method
  # ??text_encoder.forward # basically a wrapper
  # ??text_encoder.text_model.forward
def get_output_embeds(input_embeddings):
  # input_embeddings is the output of text_encoder.text_model.embeddings

  bsz, seq_len = input_embeddings.shape[:2]
  # TODO: Why does CLIP's text model use a causal mask?
  # ??_create_4d_causal_attention_mask
  causal_attention_mask = _create_4d_causal_attention_mask(
      input_shape = (bsz, seq_len), # batch_size * query_len
      dtype = input_embeddings.dtype,
      device = input_embeddings.device,
  )

  # ??text_encoder.text_model.encoder.forward
  encoder_outputs = text_encoder.text_model.encoder(
      inputs_embeds=input_embeddings,
      attention_mask=None, # We aren't using an attention mask, so that can be None
      causal_attention_mask=causal_attention_mask.to(torch_device),
      output_attentions=None,
      output_hidden_states=True, # We want the output emb,s not the final(pooled) output
  )
  output = encoder_outputs.last_hidden_state

  # There is a final layer norm we need to pass these through
  output = text_encoder.text_model.final_layer_norm(output)
  return output

out_embs_test = get_output_embeds(input_embeddings)
(out_embs_test == output_embeddings).all()

# Create images from embeddings

## Token replacement

In [ ]:
token_id = 6829
print(f"token id: {token_id} | \
the token: {tokenizer.decoder.get(token_id)}")
embedding = token_emb_layer(torch.tensor(token_id, device=torch_device))
print(f"embedding shape: {embedding.shape}")

token_id = 2368
print(f"token id: {token_id} | \
the token: {tokenizer.decoder.get(token_id)}")
embedding = token_emb_layer(torch.tensor(token_id, device=torch_device))
print(f"embedding shape: {embedding.shape}")

In [ ]:
token_embeddings = token_emb_layer(input_ids)
replacement_token_embedding = token_emb_layer(
    torch.tensor(2368, device=torch_device)
)

token_embeddings[0, torch.where(input_ids[0]==6829)] =\
    replacement_token_embedding.to(torch_device)

input_embeddings = token_embeddings + position_embeddings
modified_output_embeddings = get_output_embeds(input_embeddings)

In [ ]:
def generate_with_embs(text_embeddings):

    height = 512
    width = 512
    num_inference_steps = 30
    guidance_scale = 7.5
    batch_size = 1

    uncond_input = tokenizer(
      [""] * batch_size,
      padding="max_length",
      max_length=text_input.input_ids.shape[-1],
      return_tensors="pt"
    )
    with torch.no_grad():
        uncond_embeddings = text_encoder(
            uncond_input.input_ids.to(torch_device)
        ).last_hidden_state
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    # Prep Scheduler
    scheduler.set_timesteps(num_inference_steps)

    # Prep latents
    latents = torch.randn(
    (batch_size, unet.config.in_channels, height // 8, width // 8),
    generator = torch.manual_seed(32),
    )
    latents = latents.to(torch_device).to(torch.bfloat16)
    latents = latents * scheduler.init_noise_sigma

    # Loop
    latents = denoise_loop(scheduler, latents, text_embeddings)

    return latents_to_pil(latents)[0]

In [ ]:
generate_with_embs(modified_output_embeddings)

## Token interpolation

In [ ]:
prompt = 'skunk'
print('tokenizer(prompt):', tokenizer(prompt, add_special_tokens=False))

In [ ]:
token_embeddings = token_emb_layer(input_ids)

# Which is now a mixture of the token embeddings for 'puppy' and 'skunk.'
puppy_token_embedding = token_emb_layer(
  torch.tensor(6829, device=torch_device)
)
skunk_token_embedding = token_emb_layer(
  torch.tensor(42194, device=torch_device)
)
replacement_token_embedding = (0.5*puppy_token_embedding+
                               0.5*skunk_token_embedding)

# Insert this into the token embeddings (
token_embeddings[0, torch.where(input_ids[0]==6829)]=\
  replacement_token_embedding.to(torch_device)

input_embeddings = token_embeddings + position_embeddings
modified_output_embeddings = get_output_embeds(input_embeddings)
generate_with_embs(modified_output_embeddings)

## Textual Inversion

In [ ]:
main_path = "https://huggingface.co/sd-concepts-library"
url = f"{main_path}/birb-style/resolve/main/learned_embeds.bin"
!wget {url}

In [ ]:
birb_embed = torch.load('learned_embeds.bin')
print(birb_embed.keys())
print(birb_embed['<birb-style>'].shape)

In [ ]:
token_embeddings = token_emb_layer(input_ids)
replacement_token_embedding = birb_embed['<birb-style>'].to(torch_device).to(torch.bfloat16)
token_embeddings[0, torch.where(input_ids[0]==6829)]=\
  replacement_token_embedding.to(torch_device)
input_embeddings = token_embeddings + position_embeddings
modified_output_embeddings = get_output_embeds(input_embeddings)
generate_with_embs(modified_output_embeddings)

## Prompt interpolation

In [ ]:
text_input1 = tokenizer(
  ["A mouse"], padding="max_length",
  max_length=tokenizer.model_max_length,
  truncation=True, return_tensors="pt"
)
text_input2 = tokenizer(
  ["A leopard"], padding="max_length",
  max_length=tokenizer.model_max_length,
  truncation=True, return_tensors="pt"
)
with torch.no_grad():
    text_embeddings1 = text_encoder(
      text_input1.input_ids.to(torch_device)
    )[0]
    text_embeddings2 = text_encoder(
      text_input2.input_ids.to(torch_device)
    )[0]

mix_factor = 0.35
mixed_embeddings = (text_embeddings1*mix_factor + \
                   text_embeddings2*(1-mix_factor))
generate_with_embs(mixed_embeddings)

# Looking inside the denoising loop

- The Unet takes in the noisy latents (x) and predicts the noise.
- We use a conditional model that also takes in the timestep (`t`) and our text embedding (aka `encoder_hidden_states`) as conditioning.
- Feeding all of these into the model looks like this: `noise_pred = unet(latents, t, encoder_hidden_states=text_embeddings)["sample"]`

## Gradual vs instant noise reduction

- Given a set of noisy latents, the model predicts the noise component.
- We have these two options among others
    - Remove this noise from the noisy latents to obtain the output image instantly (`latents_x0 = latents - sigma * noise_pred`).
    - Add most of the noise back to this predicted output to get the (slightly less noisy, hopefully) input for the next diffusion step.
- To visualize this, let's generate another image, saving both the predicted output (x0) and the next step (xt-1) after every step:

In [ ]:
# Make a folder to store results
!rm -rf steps/
!mkdir -p steps/

In [ ]:
prompt = 'Oil painting of an otter in a top hat'
text_input = tokenizer(
    [prompt], padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True, return_tensors="pt"
)
with torch.no_grad():
    text_embeddings = text_encoder(
        text_input.input_ids.to(torch_device)
    ).last_hidden_state
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size,
    padding="max_length",
    max_length=max_length, return_tensors="pt"
)
with torch.no_grad():
    uncond_embeddings = text_encoder(
        uncond_input.input_ids.to(torch_device)
    ).last_hidden_state

text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
scheduler.set_timesteps(num_inference_steps)
latents = torch.randn(
  (batch_size, unet.config.in_channels, height // 8, width // 8),
  generator = torch.manual_seed(32),
)
latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma

In [ ]:
def denoise_loop(scheduler, latents, text_embeddings, start_step= 0, saved_steps_dir=None):
    with autocast(torch_device, dtype=torch.bfloat16):
        for i, t in enumerate(tqdm(scheduler.timesteps)):
            if i >= start_step:
                latent_model_input = torch.cat([latents] * 2)
                latent_model_input = scheduler.scale_model_input(latent_model_input, t)
                with torch.no_grad():
                    noise_pred = unet(
                        latent_model_input,
                        timestep = t,
                        encoder_hidden_states=text_embeddings
                    ).sample
                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                noise_pred = (
                    noise_pred_uncond
                    + guidance_scale * (noise_pred_text - noise_pred_uncond)
                )
                scheduler_step = scheduler.step(noise_pred, t, latents)
                latents = scheduler_step.prev_sample
                if saved_steps_dir:
                    latents_x0 = scheduler_step.pred_original_sample
                    im_t0 = latents_to_pil(latents_x0)[0]
                    im_next = latents_to_pil(latents)[0]
                    im = Image.new('RGB', (1024, 512))
                    im.paste(im_next, (0, 0))
                    im.paste(im_t0, (512, 0)) # right section shows x0
                    im.save(f'{saved_steps_dir}/{i:04}.jpeg')
    return latents

In [ ]:
latents = denoise_loop(scheduler, latents, text_embeddings, saved_steps_dir="steps")

In [ ]:
# Make and show the progress video (change width to 1024 for full res)
!ffmpeg -v 1 -y -f image2 -framerate 12 -i steps/%04d.jpeg -c:v libx264 -preset slow -qp 18 -pix_fmt yuv420p out.mp4
mp4 = open('out.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=600 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [ ]:
print(latents.dtype)
# latents_to_pil requires bf16 dtype
# it works fine inside the autocast context manager
# but denoise_loop doesn't change the dtype to bf16 automatically.

## Sampling

There is still some complexity hidden from us inside `latents = scheduler.step(noise_pred, i, latents)["prev_sample"]`.
- How exactly does the sampler go from the current noisy latents to a slightly less noisy version?
- Why don't we just use the model in a single step?
- Are there other ways to view this?

1. The model tries to predict the noise in an image.
2. For low noise values, we assume it does a pretty good job. For higher noise levels, it has a hard task
3. So instead of producing a perfect image, the results tend to look like a blurry mess - see the start of the video above for a visual!
4. So, samplers use the model predictions to move a small amount towards the model prediction (removing some of the noise) and then get another prediction based on this marginally-less-rubbish input, and hope that this iteratively improves the result.
5. Different samplers do this in different ways. You can try to inspect the code for the default LMS sampler by running `??scheduler.step`

---
Two videos explaining intuitively how sampling works
- the original video for this notebook (fastai team) --> [Lesson 9A 2022](https://www.youtube.com/watch?v=0_BBRNYInx8) | 24:00 to 36:00
- [But how do AI images and videos actually work? | Guest video by Welch Labs](https://www.youtube.com/watch?v=iv-5mZ_9CPY) | from 11:45 onwards

## User-defined Guidance
How can we add some extra control to this generation process?
- At each step, we're going to use our model as before to predict the noise component of x.
- Then we'll use this to produce a predicted output image and apply some **loss function** to this image.
- This function can be anything, but let's demo with a super simple example.
    - If we want images that have a lot of blue, we can craft a loss function that gives a high loss if pixels have a low blue component
- During each update step,
    - we find the gradient of the loss with respect to the current noisy latents,
    - and tweak them in the direction that reduces this loss,
    - as well as performing the normal update step

In [ ]:
def blue_loss(images):
    # How far are the blue channel values from 0.9:
    error = torch.abs(images[:,2] - 0.9).mean()
    # [:,2] -> all images in batch, only the blue channel
    return error

### Grad calculation optimizations
`latents = latents.detach().requires_grad_()`
- This gives you gradients only w.r.t. the current latents,
- not the entire calculation history graph.
- If you did not detach the latents before `requires_grad_()`:
    1. The graph would include all previous UNet forward passes
    2. Backprop would attempt to compute gradients through the entire diffusion history
    3. UNet weights (although wrapped in no_grad) and scheduler ops would be included incorrectly
    4. You'd get massive memory usage and likely crashes

`latents = latents.detach() - cond_grad * sigma**2`
- This keeps the diffusion loop cheap and stateless
- from the perspective of autograd, Without `detach()`, that new latent would still carry:
    1. The gradient relationship to the grad
    1. The gradient relationship to the loss
    1. The gradient relationship to the VAE decode

NB: We should set latents requires_grad=True **before** we do the forward pass of the unet (removing `with torch.no_grad()`) if we want mode accurate gradients. BUT this requires a lot of extra memory. You'll see both approaches used depending on whose implementation you're looking at.

In [ ]:
prompt = 'A campfire (oil on canvas)'
height = 512
width = 512
num_inference_steps = 50  
guidance_scale = 8 
batch_size = 1

text_input = tokenizer(
    [prompt], padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True, return_tensors="pt"
)
input_ids = text_input.input_ids.to(torch_device)
with torch.no_grad(): text_embeddings = text_encoder(input_ids).last_hidden_state
max_length = input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)
uc_input_ids = uncond_input.input_ids.to(torch_device)
with torch.no_grad(): uncond_embeddings = text_encoder(uc_input_ids).last_hidden_state
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

scheduler.set_timesteps(num_inference_steps)
latents = torch.randn(
  (batch_size, unet.config.in_channels, height // 8, width // 8),
  generator = torch.manual_seed(32),
)
latents = latents.to(torch_device).to(torch.bfloat16)
latents = latents * scheduler.init_noise_sigma

In [ ]:
def denoise_loop(scheduler, latents, text_embeddings, loss_fn=None, start_step= 0, saved_steps_dir=None, loss_scale=200):
    with autocast(torch_device, dtype=torch.bfloat16):
        for i, t in enumerate(tqdm(scheduler.timesteps)):
            if i >= start_step:
                latent_model_input = torch.cat([latents] * 2)
                latent_model_input = scheduler.scale_model_input(latent_model_input, t)
                with torch.no_grad():
                    noise_pred = unet(latent_model_input,timestep = t,
                        encoder_hidden_states=text_embeddings).sample
                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                noise_pred = (
                    noise_pred_uncond
                    + guidance_scale * (noise_pred_text - noise_pred_uncond)
                )
                
                if loss_fn and (i%5 == 0):
                    sigma = scheduler.sigmas[i]
                    latents = latents.detach().requires_grad_()
                    latents_x0 = latents - sigma * noise_pred
            
                    # Decode to image space
                    denoised_images = vae.decode((1 / 0.18215) * latents_x0).sample / 2 + 0.5
                    # Calculate loss
                    loss = loss_fn(denoised_images) * loss_scale
                    print(i, 'loss:', loss.item())
                    cond_grad = torch.autograd.grad(loss, latents)[0]
            
                    # Modify the latents based on this gradient
                    latents = latents.detach() - cond_grad * sigma**2
            
                scheduler_step = scheduler.step(noise_pred, t, latents)
                latents = scheduler_step.prev_sample

                if saved_steps_dir:
                    latents_x0 = scheduler_step.pred_original_sample
                    im_t0 = latents_to_pil(latents_x0)[0]
                    im_next = latents_to_pil(latents)[0]
                    im = Image.new('RGB', (1024, 512))
                    im.paste(im_next, (0, 0))
                    im.paste(im_t0, (512, 0)) # right section shows x0
                    im.save(f'{saved_steps_dir}/{i:04}.jpeg')
    return latents

In [ ]:
latents = denoise_loop(scheduler, latents, text_embeddings, blue_loss)

In [ ]:
latents_to_pil(latents)[0]